# 04b Re-chunk + Re-embed (Smaller Chunks)

**Why:** Evaluation showed context_recall=0 for 11/30 questions.
Root cause: chunk size 2000 chars splits specific facts across chunk boundaries.

**Fix:** Smaller, denser chunks with higher overlap ratio.

| Setting | Before | After |
|---|---|---|
| Chunk size | 2000 chars | 800 chars |
| Overlap | 400 chars (20%) | 300 chars (37%) |
| Est. chunks | ~2,098 | ~5,000-6,000 |

Smaller chunks = more precise retrieval of specific facts (e.g. "2.0-2.2 g/kg on deficit")

In [3]:
import json, os, re, time
from pathlib import Path
from datetime import datetime
import statistics
import numpy as np
import faiss
import warnings; warnings.filterwarnings('ignore')

from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm
load_dotenv(override=True)

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
CHUNKS_DIR = PROCESSED_DIR / 'chunks'
VS_DIR = BACKEND_DIR / 'data' / 'vectorstore'

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
EMBED_MODEL = os.getenv('EMBEDDING_MODEL')
EMBED_DIM = 3072

# New chunking config
CHUNK_SIZE = 800    # chars (~200 tokens) — was 2000
CHUNK_OVERLAP = 300    # chars (37% overlap) — was 400
BATCH_SIZE = 100

print(f'Chunk size: {CHUNK_SIZE} chars (~{CHUNK_SIZE//4} tokens)')
print(f'Overlap: {CHUNK_OVERLAP} chars ({CHUNK_OVERLAP/CHUNK_SIZE*100:.0f}%)')

Chunk size: 800 chars (~200 tokens)
Overlap: 300 chars (38%)


## 1. Load Original Extracted Files

In [4]:
PDF_DIR = PROCESSED_DIR / 'pdf_extracted'
EXCEL_DIR = PROCESSED_DIR / 'excel_extracted'
DOCX_DIR = PROCESSED_DIR / 'docx_extracted'

VISUAL_REFERENCE_FILES = {'Body fat percentage visual reference guide PTC.pdf'}
CALCULATOR_EXCEL_FILES = {
    '1RM Calculator MennoHenselmans.com.xlsx',
    'Henselmans Energy intake calculator.xlsx',
    'Henselmans Energy Balance Calculator.xlsx',
    'Training volume calculator MennoHenselmans.com.xlsx',
}

def load_all_docs() -> list:
    docs = []
    for path in sorted(PDF_DIR.glob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        fname = data['filename']
        if fname in VISUAL_REFERENCE_FILES: continue
        if not data.get('full_text','').strip(): continue
        is_cs = any(kw in fname.lower() for kw in ['case study','lifestyle case'])
        docs.append({'filename':fname,'source':data['source'],'file_type':'pdf',
                     'full_text':data['full_text'],'is_case_study':is_cs})
    for path in sorted(EXCEL_DIR.glob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        fname = data['filename']
        if fname in CALCULATOR_EXCEL_FILES: continue
        if not data.get('full_text','').strip(): continue
        is_cs = any(kw in fname.lower() for kw in ['case study','national','female powerlifter','ad libitum'])
        docs.append({'filename':fname,'source':data['source'],'file_type':'excel',
                     'full_text':data['full_text'],'is_case_study':is_cs})
    for path in sorted(DOCX_DIR.glob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        if not data.get('full_text','').strip(): continue
        is_cs = 'case study' in data['filename'].lower()
        docs.append({'filename':data['filename'],'source':data['source'],'file_type':'docx',
                     'full_text':data['full_text'],'is_case_study':is_cs})
    return docs

docs = load_all_docs()
print(f'Docs loaded: {len(docs)}')
total_chars = sum(len(d['full_text']) for d in docs)
est_chunks = total_chars // (CHUNK_SIZE - CHUNK_OVERLAP)
print(f'Total chars : {total_chars:,}')
print(f'Est. chunks : ~{est_chunks:,} (at {CHUNK_SIZE} size, {CHUNK_OVERLAP} overlap)')

Docs loaded: 83
Total chars : 3,593,771
Est. chunks : ~7,187 (at 800 size, 300 overlap)


## 2. Splitter + Cleaning

In [5]:
def is_toc_chunk(text: str) -> bool:
    return sum(1 for line in text.split('\n') if '.....' in line) >= 3

def is_garbage_chunk(text: str) -> bool:
    cleaned = re.sub(r'--- Page \d+ ---', '', text).strip()
    return len(cleaned) < 80

def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    separators = ['\n\n', '\n', '. ', '? ', '! ', ' ', '']

    def _split(text, seps):
        if not text.strip(): return []
        if len(text) <= chunk_size: return [text.strip()]
        sep = seps[0]
        rest = seps[1:]
        if sep == '':
            return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size-overlap)]
        parts = text.split(sep)
        chunks, current = [], ''
        for part in parts:
            candidate = current + sep + part if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    if len(current) > chunk_size and rest:
                        chunks.extend(_split(current, rest))
                    else:
                        chunks.append(current.strip())
                current = part
        if current.strip():
            if len(current) > chunk_size and rest:
                chunks.extend(_split(current, rest))
            else:
                chunks.append(current.strip())
        return [c for c in chunks if c.strip()]

    raw = _split(text, separators)
    if len(raw) <= 1 or overlap == 0: return raw
    overlapped = [raw[0]]
    for i in range(1, len(raw)):
        prev_tail = raw[i-1][-overlap:]
        overlapped.append(prev_tail + ' ' + raw[i])
    return overlapped

print('Splitter defined')

Splitter defined


## 3. Metadata Tagger

In [6]:
TOPIC_KEYWORDS = {
    'training': ['progressive overload','hypertrophy','strength','rep range','volume',
                 'frequency','periodization','exercise selection','program','workout',
                 'deload','intensity','RIR','RPE','failure','compound','isolation',
                 'squat','deadlift','bench','press','sets','reps'],
    'nutrition': ['protein','calorie','macro','carbohydrate','fat','diet','food',
                  'meal','intake','deficit','surplus','TDEE','BMR','energy balance',
                  'supplement','creatine','amino acid','nutrient','protein synthesis',
                  'meal frequency','leucine'],
    'recovery': ['sleep','recovery','deload','fatigue','stress','cortisol',
                 'DOMS','soreness','rest','overtraining','overreaching','adaptation'],
    'body_composition': ['body fat','lean mass','muscle mass','BF%','recomposition',
                         'bulk','cut','bulking','cutting','fat loss','muscle gain'],
    'program_design': ['program design','programming','split','PPL','upper lower',
                       'full body','mesocycle','case study','client','intake form'],
    'health': ['health','injury','pain','rehabilitation','lifestyle','adherence',
               'motivation','psychology','hormone','testosterone','insulin'],
}
LEVEL_KEYWORDS = {
    'novice':['novice','beginner','untrained','new to','starting'],
    'intermediate':['intermediate','moderate experience'],
    'advanced':['advanced','experienced','elite','competitive'],
}
GOAL_KEYWORDS = {
    'bulk':['bulk','bulking','muscle gain','lean bulk','surplus'],
    'cut':['cut','cutting','fat loss','weight loss','deficit','lean out'],
    'maintain':['maintain','maintenance','recomposition','recomp'],
}
CALCULATOR_KEYWORDS = {
    'energy_intake':['TDEE','BMR','calorie','energy intake'],
    'one_rep_max':['1RM','one rep max','working weight'],
    'training_volume':['sets per week','MEV','MAV','MRV'],
    'energy_balance':['energy balance','body composition change'],
}

def tag_metadata(text: str, source: str, is_case_study: bool) -> dict:
    t = text.lower()
    scores = {cat: sum(1 for kw in kws if kw.lower() in t) for cat, kws in TOPIC_KEYWORDS.items()}
    topic = max(scores, key=scores.get)
    if scores[topic] == 0: topic = 'general'
    level = next((lv for lv, kws in LEVEL_KEYWORDS.items() if any(kw in t for kw in kws)), 'all')
    goals = [g for g, kws in GOAL_KEYWORDS.items() if any(kw in t for kw in kws)]
    goal = goals[0] if len(goals)==1 else ('all' if not goals else 'multiple')
    calc = next((c for c, kws in CALCULATOR_KEYWORDS.items() if any(kw.lower() in t for kw in kws)), None)
    return {'source':source,'topic_category':topic,'applies_to_level':level,
            'goal_relevance':goal,'is_case_study':is_case_study,'calculator_context':calc,'is_calculator_tool':False}

print('Tagger defined')

Tagger defined


## 4. Chunk All Documents

In [7]:
all_chunks = []
chunk_stats = []

for doc in tqdm(docs, desc='Chunking'):
    fname = doc['filename']
    text = doc['full_text']
    source = doc['source']
    ftype = doc['file_type']
    is_cs = doc['is_case_study']
    stem = Path(source).stem

    raw_chunks = split_text(text)
    file_chunks = 0

    for j, chunk_text in enumerate(raw_chunks):
        if not chunk_text.strip(): continue
        if is_toc_chunk(chunk_text): continue
        if is_garbage_chunk(chunk_text): continue
        meta = tag_metadata(chunk_text, source, is_cs)
        chunk_id = f'{stem}__{j:05d}'
        all_chunks.append({'chunk_id':chunk_id,'source':source,'file_type':ftype,
                            'text':chunk_text,'metadata':meta})
        file_chunks += 1

    chunk_stats.append({'filename':fname,'chunks':file_chunks,'chars':len(text)})

# Add calculator tool descriptions
CALCULATOR_DESCRIPTIONS = [
    {'chunk_id':'calculator__one_rep_max__description','source':'1RM Calculator MennoHenselmans.com.xlsx','file_type':'excel',
     'text':'One Rep Max (1RM) Calculator — Henselmans Method\n\nEstimates 1RM from working set using Epley formula: 1RM = weight x (1 + reps/30). Call calculate_1rm(weight, reps) from calculators.py. For bodyweight exercises use calculate_bodyweight_1rm().',
     'metadata':{'source':'1RM Calculator MennoHenselmans.com.xlsx','topic_category':'training','applies_to_level':'all','goal_relevance':'all','is_case_study':False,'calculator_context':'one_rep_max','is_calculator_tool':True}},
    {'chunk_id':'calculator__energy_intake__description','source':'Henselmans Energy intake calculator.xlsx','file_type':'excel',
     'text':'Energy Intake Calculator — BMR, TDEE, Macros (Henselmans)\n\nBMR via Katch-McArdle: 370 + 21.6 x LBM kg. Activity multipliers: sedentary 1.2 to very active 1.9. Goal adjustments: bulk +250, maintain 0, cut -500, aggressive cut -750 kcal. Protein: 2.0 g/kg cut, 1.8 g/kg bulk. Fat min 0.8 g/kg. Call calculate_energy_intake() from calculators.py.',
     'metadata':{'source':'Henselmans Energy intake calculator.xlsx','topic_category':'nutrition','applies_to_level':'all','goal_relevance':'all','is_case_study':False,'calculator_context':'energy_intake','is_calculator_tool':True}},
    {'chunk_id':'calculator__energy_balance__description','source':'Henselmans Energy Balance Calculator.xlsx','file_type':'excel',
     'text':'Energy Balance Calculator — Expected Body Composition Change (Henselmans)\n\nLBM = 1,817 kcal/kg. Fat = 9,081 kcal/kg. Weekly balance = (LBM change x 1817) + (FM change x 9081). Call calculate_energy_balance() from calculators.py.',
     'metadata':{'source':'Henselmans Energy Balance Calculator.xlsx','topic_category':'nutrition','applies_to_level':'all','goal_relevance':'all','is_case_study':False,'calculator_context':'energy_balance','is_calculator_tool':True}},
    {'chunk_id':'calculator__training_volume__description','source':'Training volume calculator MennoHenselmans.com.xlsx','file_type':'excel',
     'text':'Training Volume Calculator — Optimal Sets Per Week (Henselmans)\n\nNovice/Intermediate/Advanced sets/week: Chest 10/12/16, Back 10/14/18, Shoulders 8/12/16, Biceps 6/10/14, Triceps 6/10/14, Quads 10/14/18, Hamstrings 8/10/14, Glutes 8/12/16. Female lower body +20%. Priority muscles +4 sets. Call calculate_optimal_volume() from calculators.py.',
     'metadata':{'source':'Training volume calculator MennoHenselmans.com.xlsx','topic_category':'training','applies_to_level':'all','goal_relevance':'all','is_case_study':False,'calculator_context':'training_volume','is_calculator_tool':True}},
]
all_chunks.extend(CALCULATOR_DESCRIPTIONS)

lengths = [len(c['text']) for c in all_chunks]
print(f'\nTotal chunks: {len(all_chunks):,}')
print(f'  Min length: {min(lengths)}')
print(f'  Max length: {max(lengths)}')
print(f'  Mean length: {statistics.mean(lengths):.0f}')
print(f'  Median: {statistics.median(lengths):.0f}')

Chunking:   0%|          | 0/83 [00:00<?, ?it/s]

Chunking: 100%|██████████| 83/83 [00:00<00:00, 125.54it/s]



Total chunks: 6,077
  Min length: 80
  Max length: 1101
  Mean length: 843
  Median: 901


## 5. Save Clean Chunks

In [8]:
output = {'created_at':datetime.now().isoformat(),'total_chunks':len(all_chunks),
           'chunk_size_chars':CHUNK_SIZE,'chunk_overlap_chars':CHUNK_OVERLAP,'chunks':all_chunks}
out_path = CHUNKS_DIR / 'clean_chunks_v2.json'
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved: {out_path}')
print(f'Size : {out_path.stat().st_size/(1024*1024):.1f} MB')

Saved: D:\Cybernetic Gym Assistant\backend\data\processed\chunks\clean_chunks_v2.json
Size : 7.7 MB


## 6. Embed with OpenAI

In [9]:
# texts = [c['text'] for c in all_chunks]
# est_tokens = sum(len(t) for t in texts) // 4
# est_cost = (est_tokens / 1000) * 0.00013
# print(f'Chunks to embed: {len(texts):,}')
# print(f'Est. tokens: ~{est_tokens:,}')
# print(f'Est. cost: ~${est_cost:.3f} USD')

# all_embeddings = []
# for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding'):
#     batch = texts[i:i+BATCH_SIZE]
#     response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
#     sorted_d = sorted(response.data, key=lambda x: x.index)
#     all_embeddings.extend([item.embedding for item in sorted_d])
#     time.sleep(0.05)

# embeddings = np.array(all_embeddings, dtype='float32')
# print(f'\nEmbeddings shape : {embeddings.shape}')
# assert embeddings.shape == (len(all_chunks), EMBED_DIM)
# print('Shape OK')

import gc

BATCH_SIZE = 16          # ← was 100; smaller JSON responses = no OOM on parse
N = len(all_chunks)
CHECKPOINT_PATH = VS_DIR / 'embed_checkpoint.json'

# Pre-allocate on disk - never fully in RAM
emb_path = VS_DIR / 'embeddings_v2.npy'
embeddings = np.lib.format.open_memmap(
    str(emb_path), mode='w+', dtype='float32', shape=(N, EMBED_DIM)
)

# Resume support - safe to re-run after a crash
done_up_to = 0
if CHECKPOINT_PATH.exists():
    done_up_to = json.loads(CHECKPOINT_PATH.read_text())['done_up_to']
    embeddings = np.lib.format.open_memmap(
        str(emb_path), mode='r+', dtype='float32', shape=(N, EMBED_DIM)
    )
    print(f'Resuming from chunk {done_up_to}/{N}')

texts = [c['text'] for c in all_chunks]
est_tokens = sum(len(t) for t in texts) // 4
est_cost = (est_tokens / 1000) * 0.00013
print(f'Chunks to embed: {N:,}')
print(f'Est. tokens: ~{est_tokens:,}')
print(f'Est. cost: ~${est_cost:.3f} USD')

for i in tqdm(range(done_up_to, N, BATCH_SIZE), desc='Embedding'):
    batch = texts[i:i + BATCH_SIZE]
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
    sorted_d = sorted(response.data, key=lambda x: x.index)

    vecs = np.array([item.embedding for item in sorted_d], dtype='float32')
    embeddings[i:i + len(batch)] = vecs   # written straight to disk

    CHECKPOINT_PATH.write_text(json.dumps({'done_up_to': i + len(batch)}))
    del response, sorted_d, vecs
    gc.collect()
    time.sleep(0.05)

embeddings.flush()
CHECKPOINT_PATH.unlink(missing_ok=True)   # clean up on success

print(f'\nEmbeddings shape : {embeddings.shape}')
assert embeddings.shape == (N, EMBED_DIM)
print('Shape OK')


Chunks to embed: 6,077
Est. tokens: ~1,280,763
Est. cost: ~$0.166 USD


Embedding: 100%|██████████| 380/380 [02:36<00:00,  2.43it/s]


Embeddings shape : (6077, 3072)
Shape OK


## 7. Build FAISS Index + Save Metadata

In [10]:
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

index_path = VS_DIR / 'henselmans_openai.index'
faiss.write_index(index, str(index_path))
print(f'FAISS index: {index.ntotal:,} vectors -> {index_path}')
print(f'Index size: {index_path.stat().st_size/(1024*1024):.1f} MB')

metadata = []
for i, c in enumerate(all_chunks):
    metadata.append({
        'faiss_id':i,'chunk_id':c['chunk_id'],'source':c['source'],
        'file_type':c['file_type'],'text':c['text'],
        'topic_category':c['metadata']['topic_category'],
        'applies_to_level':c['metadata']['applies_to_level'],
        'goal_relevance':c['metadata']['goal_relevance'],
        'is_case_study':c['metadata']['is_case_study'],
        'calculator_context':c['metadata']['calculator_context'],
        'is_calculator_tool':c['metadata'].get('is_calculator_tool',False),
    })

meta_path = VS_DIR / 'henselmans_openai_metadata.json'
meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Metadata: {len(metadata):,} records -> {meta_path}')

FAISS index: 6,077 vectors -> D:\Cybernetic Gym Assistant\backend\data\vectorstore\henselmans_openai.index
Index size: 71.2 MB
Metadata: 6,077 records -> D:\Cybernetic Gym Assistant\backend\data\vectorstore\henselmans_openai_metadata.json


## 8. Quick Retrieval Test

In [11]:
def embed_query(text):
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=[text])
    emb = np.array(r.data[0].embedding, dtype='float32').reshape(1,-1)
    faiss.normalize_L2(emb)
    return emb

def retrieve(query, k=5):
    q_emb = embed_query(query)
    scores, ids = index.search(q_emb, k)
    return [{**metadata[idx],'score':float(s)} for s,idx in zip(scores[0],ids[0]) if idx!=-1]

# Test the problematic question from evaluation
print('TEST: protein intake on deficit (was recall=0)')
for r in retrieve('optimal protein intake cutting deficit 2.0 g/kg', k=5):
    print(f'  [{r["score"]:.4f}] {r["source"][:40]} | {r["text"][:120]}...')
print()
print('TEST: Bulgarian cross-lingual')
for r in retrieve('колко протеин дефицит мускули', k=5):
    print(f'  [{r["score"]:.4f}] {r["source"][:40]} | {r["text"][:120]}...')

TEST: protein intake on deficit (was recall=0)
  [0.6592] Protein PTC 2022.pdf | (1988) found 0.73 g/lb sufficed to maintain lean body mass in
cutting weightlifters.
• Helms et al. (2014) found no diff...
  [0.6389] Protein PTC 2022.pdf | ost only 200 grams less
fat than the 2.1 g/kg group (0.9 kg vs. 1.1 kg fat), but their fat loss did not reach
statistica...
  [0.6202] Protein PTC 2022.pdf | nd protein synthesis remained
unchanged.
• Jose Antonio et al. (2015) studied strength trained subjects in a mild energy...
  [0.6200] Protein PTC 2022.pdf | g
idea seems to have come from a misinterpretation of the nitrogen balance literature
showing more lean mass is lost in ...
  [0.6143] Protein PTC 2022.pdf | data to go by, 2.4-3.7 g/kg/d may be a wise range to go by, depending on the dosages
used. Muscle memory
Trainees that r...

TEST: Bulgarian cross-lingual
  [0.4078] Energy PTC 2022.pdf | energy deficit compared to maintenance energy intake without any change in muscle --- Page 58 ---
p

## 9. Summary

In [12]:
print('=' * 55)
print('  NOTEBOOK 04b — COMPLETE')
print('=' * 55)
print(f'  Chunks: {len(all_chunks):,}  (was ~2,098)')
print(f'  Chunk size: {CHUNK_SIZE} chars (was 2000)')
print(f'  Overlap: {CHUNK_OVERLAP} chars / {CHUNK_OVERLAP/CHUNK_SIZE*100:.0f}% (was 20%)')
print(f'  FAISS index: henselmans_openai.index  (REPLACED)')
print(f'  Metadata: henselmans_openai_metadata.json  (REPLACED)')


  NOTEBOOK 04b — COMPLETE
  Chunks: 6,077  (was ~2,098)
  Chunk size: 800 chars (was 2000)
  Overlap: 300 chars / 38% (was 20%)
  FAISS index: henselmans_openai.index  (REPLACED)
  Metadata: henselmans_openai_metadata.json  (REPLACED)
